In [126]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
import pandas as pd
import numpy as np
import joblib

In [127]:
df = pd.read_csv("TehranHouse.csv")
print(df.columns[df.isnull().any()].tolist())
df = df.dropna(subset=["Address"])

['Address']


In [128]:
df.info()

<class 'pandas.DataFrame'>
Index: 3456 entries, 0 to 3478
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Area        3456 non-null   str    
 1   Room        3456 non-null   int64  
 2   Parking     3456 non-null   bool   
 3   Warehouse   3456 non-null   bool   
 4   Elevator    3456 non-null   bool   
 5   Address     3456 non-null   str    
 6   Price       3456 non-null   float64
 7   Price(USD)  3456 non-null   float64
dtypes: bool(3), float64(2), int64(1), str(2)
memory usage: 172.1 KB


In [129]:
df.isna().sum()

Area          0
Room          0
Parking       0
Warehouse     0
Elevator      0
Address       0
Price         0
Price(USD)    0
dtype: int64

In [130]:
df.head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
0,63,1,True,True,True,Shahran,1.850000e+09,61666.67
1,60,1,True,True,True,Shahran,1.850000e+09,61666.67
2,79,2,True,True,True,Pardis,5.500000e+08,18333.33
3,95,2,True,True,True,Shahrake Qods,9.025000e+08,30083.33
4,123,2,True,True,True,Shahrake Gharb,7.000000e+09,233333.33


In [131]:
print(df.Room.unique())
print(df.Area.unique())

[1 2 3 0 4 5]
<StringArray>
[             '63',              '60',              '79',              '95',
             '123',              '70',              '87',              '59',
              '54',              '71',
 ...
 ' 2,550,000,000 ',             '330',             '212',             '345',
             '530',             '630',              '30',             '205',
             '285',             '186']
Length: 243, dtype: str


Clean the Area Column :
Convert `Area` to numeric values. Anything that cannot be converted becomes NaN and is then dropped.

In [132]:
def convert_area_to_num(x):
    try:
        return float(x)
    except:
        return None

In [133]:
df2 = df.copy()
df2["Area"] = df2["Area"].apply(convert_area_to_num)

In [134]:
df2[df2["Area"]>500]

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
356,650.0,2,True,True,False,Absard,5.200000e+09,173333.33
431,660.0,5,True,True,False,Lavasan,5.500000e+10,1833333.33
573,863.0,2,True,True,True,Gheitarieh,7.830000e+09,261000.00
819,680.0,5,True,True,False,Ekhtiarieh,8.160000e+10,2720000.00
831,750.0,5,True,True,True,Mahmoudieh,7.500000e+10,2500000.00
1236,600.0,2,True,True,True,Qalandari,9.150000e+09,305000.00
1407,574.0,5,True,True,False,North Program Organization,6.000000e+10,2000000.00
1694,929.0,5,True,True,False,Zafar,8.000000e+10,2666666.67
1700,550.0,3,True,True,False,Absard,4.500000e+09,150000.00
1810,705.0,5,True,True,False,Abazar,9.100000e+10,3033333.33


In [135]:
df2 = df2.dropna()

In [136]:
df2.isna().sum()

Area          0
Room          0
Parking       0
Warehouse     0
Elevator      0
Address       0
Price         0
Price(USD)    0
dtype: int64

In [137]:
df2["price_per_m"] = df2["Price(USD)"] / df2["Area"]
print(df2["price_per_m"].describe())
print(df2.sort_values("price_per_m").head(10))
print(df2.sort_values("price_per_m").tail(10))

count     3450.000000
mean      1375.807913
std       1055.277041
min          0.750000
25%        666.666700
50%       1157.651049
75%       1832.410588
max      13888.888889
Name: price_per_m, dtype: float64
       Area  Room  Parking  Warehouse  Elevator                    Address  \
136   160.0     1    False      False     False                    Qarchak   
2770   83.0     2     True       True      True                      Ozgol   
731    75.0     2     True       True      True                     Pardis   
2721  110.0     0     True       True      True                     Parand   
2201   49.0     1     True       True     False                   Andisheh   
3300  120.0     3     True       True      True  Persian Gulf Martyrs Lake   
1343   78.0     2     True       True      True                     Parand   
3149  118.0     2     True       True      True           Northern Chitgar   
1443  350.0     3     True       True     False                   Pasdaran   
1543  123.

In [138]:
df2[df2.Area/df2.Room<28].head()

,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD),price_per_m
8,54.0,2,True,True,False,Andisheh,4.930000e+08,16433.33,304.320926
776,54.0,2,True,True,False,Shahran,1.850000e+09,61666.67,1141.975370
846,51.0,2,False,True,True,Komeil,1.050000e+09,35000.00,686.274510
1371,55.0,2,False,False,False,Parastar,1.280000e+09,42666.67,775.757636
1929,50.0,2,True,True,False,Shahran,1.820000e+09,60666.67,1213.333400


In [139]:
df2.shape

(3450, 9)

Remove Unrealistic Area-per-Room Rows:
Drop rows where the area per room is below 28 m², which are unrealistic (e.g. 54 m² with 2 rooms is borderline, 160 m² for a 0.75 $/m² listing is clearly wrong).


In [140]:
df3 = df2[~(df2.Area/df2.Room<28)]

In [141]:
df3.shape

(3422, 9)

Remove Price-per-Meter Outliers per Neighborhood :
For each neighborhood, keep only rows whose price per m² is within `k` standard deviations of that neighborhood's mean. Neighborhoods are handled separately because prices differ greatly between areas.

In [142]:
def remove_ppm_outliers(df, k=2):
    g = df.groupby("Address")["price_per_m"]
    m = g.transform("mean")
    st = g.transform("std")
    return df[(df.price_per_m > m - k*st) & (df.price_per_m <= m + k*st)]
df4 = remove_ppm_outliers(df3)
df4.shape

(3238, 9)

In [143]:
X = df4.drop(columns=["Price", "Price(USD)", "price_per_m"])
y = df4["Price(USD)"]

In [144]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ("Address", )),
    ("scaler", StandardScaler(), ["Area", "Room"])
], remainder="passthrough")

In [145]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Models and Hyperparameter Grids :
Define the candidate models (SVR, Random Forest, Ridge, Decision Tree) and the hyperparameters to search for each.

In [146]:
model_params = {
    "svr": {
        "model": SVR(),
        "params": {
            "C": [1, 10, 100],
            "kernel": ["rbf", "linear"]
        }
    },
    "random forest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
        }
    },
    "ridge": {
        "model": Ridge(),
        "params": {
            "alpha": [0.1, 1, 10, 100],
        }
    },
    "tree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "criterion": ["squared_error", "absolute_error"],
            "max_depth": [None, 5, 10, 20]
        }
    }
}

Train and Compare Models :
For each model, run GridSearchCV with 5-fold cross-validation inside a pipeline. The target is log-transformed (`log1p`) to reduce skew and converted back for predictions. Record the best CV score, best parameters, and test score.

In [147]:
scores = []

for model_name, mp in model_params.items():
    grid = GridSearchCV(mp["model"], mp["params"], cv=5)
    pipe = Pipeline([("pre", pre), ("grid", grid)])
    model = TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    model.fit(X_train, y_train)
    fitted_grid = model.regressor_.named_steps["grid"]
    scores.append({
        "model": model_name,
        "best_score": fitted_grid.best_score_,
        "best_params": fitted_grid.best_params_,
        "test_score": model.score(X_test, y_test),
    })

In [148]:
scores

[{'model': 'svr',
  'best_score': np.float64(0.9074800360578473),
  'best_params': {'C': 10, 'kernel': 'rbf'},
  'test_score': 0.6837591541584722},
 {'model': 'random forest',
  'best_score': np.float64(0.8550978351769295),
  'best_params': {'n_estimators': 200},
  'test_score': 0.8132091912326134},
 {'model': 'ridge',
  'best_score': np.float64(0.8962495452274697),
  'best_params': {'alpha': 0.1},
  'test_score': -4.260372736161907},
 {'model': 'tree',
  'best_score': np.float64(0.8185968035516549),
  'best_params': {'criterion': 'squared_error', 'max_depth': None},
  'test_score': 0.5740718991075997}]